# L5b: Minimum-Variance Portfolios, the Efficient Frontier, and the Capital Allocation Line
In this lecture, we take the estimated growth-rate means and covariance matrix of L5a and ask how much of our wealth to put in each asset. The answer, due to Markowitz, is to treat a portfolio's expected growth rate as its reward and the variance of its growth rate as its risk, and to solve for the weights that minimize risk for a required reward. Doing this for every reward level traces the efficient frontier; adding a risk-free asset collapses the choice to a single risky fund, the tangent portfolio, and a straight line, the capital allocation line. We finish with the assumptions under which that tangent portfolio is the market portfolio, and with the reason estimation error can undo all of it.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Formulate portfolio reward and risk:__ Write the expected growth rate and the growth-rate variance of a portfolio as functions of the weight vector, the mean growth-rate vector, and the covariance matrix, and state what the one-period objective does and does not describe.
> * __Construct minimum-variance portfolios and the frontier:__ Derive the global minimum-variance portfolio, solve the target-growth problem with and without a long-only constraint, and distinguish the feasible set, the minimum-variance frontier, and its efficient branch.
> * __Add a risk-free asset:__ Derive the capital allocation line and the tangent portfolio, state the two-fund separation result and the constraints that break it, and explain what must hold before the tangent portfolio can be called the market portfolio.

Let's get started!
___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Compute minimum-variance portfolios, the efficient frontier, and the capital allocation line from data](CHEME-5660-L5b-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). Estimate the inputs for a chosen set of firms from the 2014 to 2024 data, compute the global minimum-variance portfolio in closed form and with a long-only solver, sweep the target growth rate to trace the efficient frontier, find the tangent portfolio and the capital allocation line, and compare the optimized portfolios with equal weights and an index fund on 2025 data the optimizer never saw.

The second example simulates the portfolios the first one computes:

> [▶ Simulate a portfolio with multiple asset GBM](CHEME-5660-L5b-Example-MAGBM-Portfolio-Fall-2026.ipynb). Estimate the multiple asset GBM model of L5a for a set of firms, simulate correlated 2025 price paths, compute the buy-and-hold wealth of the minimum-variance allocation on every path with prediction bands next to the realized path and an index fund, and evaluate the portfolio through the distribution of its scaled net present value.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Concept Review: MAGBM, the Covariance Matrix, and Portfolio Wealth
In L5a, we extended geometric Brownian motion to $M$ correlated assets $\mathcal{P}=\{1,2,\dots,M\}$ with share prices $S_{i}(t)$ at time $t$. Each asset $i$ has a mean growth rate $\bar g_{i}$ (units: inverse years), the assets share a covariance rate $\mathbf{C}\in\mathbb{R}^{M\times M}$ (units: inverse years) with a covariance factor $\mathbf{A}\mathbf{A}^{\top}=\mathbf{C}$, and one vector of independent standard normal shocks $\mathbf{Z}\sim\mathcal{N}(\mathbf{0},\mathbf{I}_{M})$, redrawn at every time step $\Delta{t}$, drives all assets through the exact one-step transition:
$$
\begin{align*}
S_{i}(t+\Delta{t}) &= S_{i}(t)\cdot\exp\Biggl[\bar g_{i}\,\Delta{t} + \sqrt{\Delta{t}}\cdot\left(\mathbf{A}\mathbf{Z}\right)_{i}\Biggr]\qquad{i\in\mathcal{P}}\\
\end{align*}
$$
Dividing by $S_{i}(t)$, taking logarithms, and dividing by $\Delta{t}$ gives the one-step growth-rate vector $\mathbf{g}=\bar{\mathbf{g}}+\mathbf{A}\mathbf{Z}/\sqrt{\Delta{t}}$, and with it the scaling rule that ties the model to data: the growth-rate covariance is $\text{Cov}(\mathbf{g})=\mathbf{C}/\Delta{t}$ (units: inverse years squared), the log-return covariance is $\mathbf{C}\Delta{t}$ (dimensionless), and $\mathbf{C}$ itself is the covariance rate. From $N$ days of growth-rate data we estimated the sample-mean vector $\mathbf{g}^{\prime}$ (our estimate of $\bar{\mathbf{g}}$) and the growth-rate covariance $\hat{\mathbf{\Sigma}}_{g}=\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}/(N-1)$ from the centered data matrix, with the covariance rate $\hat{\mathbf{C}}=\Delta{t}\,\hat{\mathbf{\Sigma}}_{g}$ and the correlation $\rho_{ij}=\hat{\Sigma}_{g,ij}/\sqrt{\hat{\Sigma}_{g,ii}\hat{\Sigma}_{g,jj}}$ (defined when both variances are positive). Two facts from L5a carry the whole lecture. First, $\hat{\mathbf{\Sigma}}_{g}$ is symmetric and positive semidefinite by construction, $\mathbf{v}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{v}=\lVert\tilde{\mathbf{G}}\mathbf{v}\rVert_{2}^{2}/(N-1)\geq0$ for every $\mathbf{v}$, so a portfolio variance can never come out negative and the minimization below is a convex problem. Second, when $N$ is large relative to $M$ (as it is for a handful of firms and eleven years of daily data), the estimate is positive definite and can be inverted; L5a explains when it is not.

If we did not finish the covariance example in L5a, we pick up its analysis now:

> __Example__
>
> [▶ Compute the covariance matrix for our dataset](../L5a/CHEME-5660-L5a-Example-CovarianceMatrix-Fall-2026.ipynb). Compute the empirical growth-rate covariance matrix for the firms in our dataset, convert it to the GBM covariance rate, verify it against built-in functions and the volatilities we estimated in L4b, and examine the covariance and correlation of a pair of firms.

L5a also defined portfolio wealth. For initial wealth $W_{0}$ and weights $\mathbf{w}$ on the long-only simplex ($w_{i}\geq0$, $\sum_{i}w_{i}=1$), buying $n_{i}=w_{i}W_{0}/S_{i}(0)$ shares of each asset and holding them gives the __buy-and-hold__ wealth $W_{t}=\sum_{i\in\mathcal{P}}n_{i}S_{i}(t)$, and the trade rule of L4b applies to it unchanged: for a benchmark growth rate $g_{b}$ and holding period $T$, the scaled NPV is $\rho_{T}=(W_{T}/W_{0})e^{-g_{b}T}-1$, and a target $\rho_{\star}$ turns it into a probability, which for a portfolio we estimate by simulation because a sum of correlated lognormals has no closed form. What L5a did not say is how to choose $\mathbf{w}$; Dirichlet sampling explored the simplex without preferring any point of it. Today we optimize.
___

## Modern Portfolio Theory: The Reward and Risk of a Portfolio
Modern Portfolio Theory (MPT) is a framework for choosing portfolio weights that maximizes expected growth rate for a given level of risk or, equivalently, minimizes risk for a given level of expected growth rate.

> __Reference:__ Modern Portfolio Theory was introduced by Harry Markowitz in the 1950s and has since become a foundational concept in finance. Markowitz was awarded the Nobel Prize in Economic Sciences in 1990 for this work. The original publication is [Portfolio Selection, The Journal of Finance, Vol. 7, No. 1 (Mar., 1952), pp. 77-91](https://www.jstor.org/stable/2975974); the Nobel Prize information is [here](https://www.nobelprize.org/prizes/economic-sciences/1990/markowitz/facts/).

Let's warm up to MPT by watching a short video that introduces the key concepts: [here](https://www.youtube.com/watch?v=VsMpw-qnPZY). Markowitz's central insight was that an asset should not be judged in isolation: a volatile asset can reduce a portfolio's variance when its growth rate moves against the other holdings, and individually stable assets provide little diversification when they move together. To make that precise we need a reward and a risk for the portfolio as a whole, both written in terms of the weights.

### Portfolio reward
Consider a portfolio $\mathcal{P}$ of $M$ risky assets held at weights $\mathbf{w}=[w_{1},\dots,w_{M}]^{\top}$, $\sum_{i}w_{i}=1$, over one period. Let $\mathbf{g}=[g_{1},\dots,g_{M}]^{\top}$ be the one-period growth rates of the assets, a random vector with mean $\bar{\mathbf{g}}=\mathbb{E}[\mathbf{g}]$ and covariance $\mathbf{\Sigma}_{g}=\text{Cov}(\mathbf{g})$. The __portfolio growth rate__ over that period is the weighted growth rate (as in L5a):
$$
\begin{align*}
g_{p} &= \sum_{i\in\mathcal{P}}w_{i}\,g_{i} = \mathbf{w}^{\top}\mathbf{g}
\end{align*}
$$
so the __reward__ of the portfolio, its expected growth rate, is the weighted average of the assets' expected growth rates:
$$
\boxed{
\begin{align*}
\mathbb{E}\left[g_{p}\right] &= \sum_{i\in\mathcal{P}}w_{i}\,\mathbb{E}\left[g_{i}\right] = \mathbf{w}^{\top}\bar{\mathbf{g}}
\end{align*}}
$$
estimated from data by $\mathbf{w}^{\top}\mathbf{g}^{\prime}$. Reward has units of inverse years.

> __What the objective is and is not:__ Over one period, a portfolio held at weights $\mathbf{w}$ has the exact simple return $R_{p}=\sum_{i}w_{i}(e^{g_{i}\Delta{t}}-1)$ and the exact log growth rate $\frac{1}{\Delta{t}}\ln\bigl(\sum_{i}w_{i}e^{g_{i}\Delta{t}}\bigr)$; expanding either to first order in $\Delta{t}$ gives $R_{p}\approx\Delta{t}\,\mathbf{w}^{\top}\mathbf{g}$ and a log growth rate of $\mathbf{w}^{\top}\mathbf{g}$. So $g_{p}=\mathbf{w}^{\top}\mathbf{g}$ is the __linearized__ one-period growth rate, accurate to first order in the time step (for daily data the neglected terms are tiny). Over many periods the log growth of buy-and-hold wealth $W_{t}$ is __not__ the weighted sum of the assets' log growth rates, because the weights drift and the logarithm of a sum is not the sum of logarithms (L5a). MPT is a statement about the linearized one-period statistic. Everything we derive today (the frontier, the tangent portfolio, the capital allocation line) inherits that scope, which is why the example evaluates the optimized weights afterwards with the buy-and-hold wealth rule, an exact and different calculation.

### Portfolio risk
The __risk__ of the portfolio is the variance of its growth rate, which by the rules for the variance of a linear combination collects the variances of the assets and every pairwise covariance:
$$
\boxed{
\begin{align*}
\text{Var}\left(g_{p}\right) &= \sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_{i}w_{j}\underbrace{\text{Cov}\left(g_{i},g_{j}\right)}_{=\;\rho_{ij}\sqrt{\Sigma_{g,ii}\Sigma_{g,jj}}} = \mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}
\end{align*}}
$$
where $\Sigma_{g,ij}=\text{Cov}(g_{i},g_{j})$ and $\rho_{ij}$ is the correlation of L5a; from data we use $\hat{\mathbf{\Sigma}}_{g}$. We write $\sigma_{g,p}=\sqrt{\text{Var}(g_{p})}$ for the portfolio's __growth-rate standard deviation__ (units: inverse years, the portfolio analogue of L3a's $\sigma_{g}$); it is not a volatility, and the volatility of the portfolio would be $\sigma_{g,p}\sqrt{\Delta{t}}$. To see what the cross terms buy, take two assets with weights $w$ and $1-w$, $0<w<1$:
$$
\begin{align*}
\text{Var}\left(g_{p}\right) &= w^{2}\Sigma_{g,11} + (1-w)^{2}\Sigma_{g,22} + 2w(1-w)\,\rho_{12}\sqrt{\Sigma_{g,11}\Sigma_{g,22}}
\end{align*}
$$
If $\rho_{12}=1$ the right-hand side is a perfect square, $\left(w\sqrt{\Sigma_{g,11}}+(1-w)\sqrt{\Sigma_{g,22}}\right)^{2}$, and the portfolio's standard deviation is the weighted average of the assets': mixing buys nothing. For every $\rho_{12}<1$ the cross term is smaller and the standard deviation falls below the weighted average; and if $\rho_{12}$ is below the ratio of the smaller to the larger of the two standard deviations, some mixture has less variance than either asset alone. That is __diversification__, and it is a statement about covariance, not about the number of holdings.

> __Growth rates versus returns:__ Everything above can be written with one-period log returns $\mathbf{r}=\Delta{t}\,\mathbf{g}$ instead: $\mathbb{E}[r_{p}]=\Delta{t}\,\mathbb{E}[g_{p}]$ (dimensionless) and $\text{Var}(r_{p})=\Delta{t}^{2}\,\text{Var}(g_{p})$, with the log-return covariance $\mathbf{\Sigma}_{r}=\Delta{t}^{2}\mathbf{\Sigma}_{g}$ (L5a's scaling rule). The optimization does not care which we use: multiplying the covariance by any positive constant, and the means and the target by any (other) positive constant, leaves the minimizing weights unchanged. What does change is every reported number, so the units of the risk axis, the growth axis, and the Sharpe ratio (defined below) must be stated. For daily data $\hat{\mathbf{\Sigma}}_{g}=252\,\hat{\mathbf{C}}$: growth-rate variances look large next to conventional annual covariance-rate entries, and that factor is the unit conversion, not an estimation failure.

Ok, we have our reward and risk measures for a portfolio of $M$ assets. Now let's use them to construct a portfolio that balances the two.
___

## Minimum-Variance Portfolios
The goal of MPT is to find the weights $\mathbf{w}$ that minimize risk for a required reward. We build up in three steps: no reward requirement at all, then a long-only constraint, then a target growth rate.

### The global minimum-variance portfolio
The __global minimum-variance (GMV) portfolio__ is the fully invested portfolio with the smallest variance, with no requirement on its expected growth rate. When short positions are allowed and only the budget constraint is imposed, it has a closed form. Let $\mathbf{\Sigma}_{g}$ be symmetric positive definite and $\mathbf{1}\in\mathbb{R}^{M}$ the vector of ones; the problem and its solution are:
$$
\boxed{
\begin{align*}
\min_{\mathbf{w}}\;\tfrac{1}{2}\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}\quad\text{subject to}\quad\mathbf{1}^{\top}\mathbf{w}=1
\quad\Longrightarrow\quad
\mathbf{w}_{\text{GMV}} = \frac{\mathbf{\Sigma}_{g}^{-1}\mathbf{1}}{\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\mathbf{1}},\qquad
\sigma^{2}_{g,\text{GMV}} = \frac{1}{\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\mathbf{1}}\quad\blacksquare
\end{align*}}
$$
The factor of one half is a convenience that cancels in the derivation and does not change the minimizer.

> __Derivation:__ Introduce a Lagrange multiplier $\lambda\in\mathbb{R}$ for the budget constraint and form the Lagrangian:
> $$\mathcal{L}(\mathbf{w},\lambda) = \tfrac{1}{2}\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w} - \lambda\left(\mathbf{1}^{\top}\mathbf{w}-1\right)$$
> Setting the gradient with respect to $\mathbf{w}$ to zero gives $\mathbf{\Sigma}_{g}\mathbf{w}-\lambda\mathbf{1}=\mathbf{0}$, so $\mathbf{w}=\lambda\mathbf{\Sigma}_{g}^{-1}\mathbf{1}$; enforcing $\mathbf{1}^{\top}\mathbf{w}=1$ gives $\lambda=(\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\mathbf{1})^{-1}$ and the boxed weights. Substituting back, $\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}=\lambda^{2}\,\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\mathbf{1}=\lambda$, which is the boxed variance. Because $\mathbf{\Sigma}_{g}$ is positive definite the objective is strictly convex, so the stationary point is the unique minimum. $\blacksquare$

Nothing in the closed form keeps the weights non-negative; the example shows the GMV portfolio for a set of thirteen firms shorting several of the volatile names to hedge the others.

### The long-only problem
Forbidding short positions adds the constraints $w_{i}\geq0$ for all $i\in\mathcal{P}$ (with $\sum_{i}w_{i}=1$ this also rules out leverage: no position exceeds the budget). The problem is now a __convex quadratic program__: the objective is a positive semidefinite quadratic and the constraints are linear. Its minimizer may sit on a boundary with some weights exactly zero, the closed form above no longer applies, and we solve it numerically. The course package sets it up in [JuMP](https://jump.dev/JuMP.jl/stable/) and solves it with an interior-point method:
$$
\boxed{
\begin{align*}
\text{minimize}~&\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{w}\\
\text{subject to}~&\hat{\mathbf{g}}^{\top}\mathbf{w}\geq g_{\star}\\
&\mathbf{1}^{\top}\mathbf{w}=1\\
&0\leq w_{i}\leq 1\qquad\forall{i}\in\mathcal{P}
\end{align*}}
$$
where $\hat{\mathbf{g}}$ is the estimated mean growth-rate vector and $g_{\star}$ is a __target growth rate__ chosen by the investor. Notice that the target enters as a floor. Set $g_{\star}$ at or below the smallest single-asset mean growth rate and every long-only portfolio satisfies the constraint, so the solver returns the long-only GMV portfolio; its variance can never be smaller than the closed form's, because the long-only feasible set is a subset of the unconstrained one. Other realistic constraints (position limits, turnover, round lots, taxes, transaction costs) change the feasible set further and are added the same way.

### The target-growth problem and the efficient frontier
Now vary the target. The classical formulation writes the growth requirement as an equality, $\bar{\mathbf{g}}^{\top}\mathbf{w}=g_{\star}$, and for each $g_{\star}$ finds the smallest variance that reaches it exactly; the set of these portfolios, indexed by $g_{\star}$, is the __minimum-variance frontier__. With short positions allowed it too has a closed form: writing $a=\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\mathbf{1}$, $b=\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\bar{\mathbf{g}}$, $c=\bar{\mathbf{g}}^{\top}\mathbf{\Sigma}_{g}^{-1}\bar{\mathbf{g}}$, and $d=ac-b^{2}$ (positive whenever $\bar{\mathbf{g}}$ is not a multiple of $\mathbf{1}$, that is, whenever the assets do not all have the same mean growth rate), the frontier is the curve:
$$
\begin{align*}
\sigma_{g}^{2}(g_{\star}) &= \frac{a\,g_{\star}^{2}-2b\,g_{\star}+c}{d}
\end{align*}
$$
a hyperbola in the $(\sigma_{g},g)$ plane whose vertex is the GMV portfolio (at $g_{\star}=b/a$, with $\sigma_{g}^{2}=1/a$, the same value as before). Every point on the hyperbola is a minimum-variance portfolio for its growth rate, but only the __upper branch__, the portfolios with expected growth rate at or above the GMV portfolio's, is __efficient__: for each of them no other feasible portfolio has both at least as much expected growth and no more variance, with one strict improvement. A portfolio on the lower branch is dominated by the point directly above it on the upper branch, which has the same variance and more growth. Every fully invested portfolio lands on or to the right of the hyperbola in the $(\sigma_{g},g)$ plane; that __attainable region__ is what the weight vectors reach, and the frontier is its left boundary.

The package's inequality formulation sees the same picture from a different angle. Because the target is a floor, every $g_{\star}$ below the GMV growth rate returns the GMV portfolio itself, and every $g_{\star}$ above it returns the efficient portfolio at that growth rate, so a sweep of the floor traces the GMV portfolio and the efficient branch, and never the lower branch. Under the long-only constraint the sweep ends at the largest single-asset mean growth rate, the most a long-only portfolio can promise, and the long-only frontier lies on or to the right of the unconstrained hyperbola, touching it only where the unconstrained solution happens to be long-only. The figure sketches the picture: the frontier through three portfolios, the GMV portfolio at the vertex, the efficient upper branch, and the dominated lower branch.

<div>
    <center>
        <img src="figs/Fig-MinVar-Portfolio-RA-Schematic.png" width="620" alt="Schematic of the minimum-variance frontier in the risk-growth plane: a curve opening to the right with the global minimum-variance portfolio at its vertex, the efficient upper branch drawn solid through a portfolio labeled 1, the dominated lower branch dashed through a portfolio labeled 3 with the same risk as portfolio 1 and lower expected growth, and the GMV portfolio labeled 2"/>
    </center>
</div>

To construct the efficient frontier from data, we estimate $\hat{\mathbf{g}}$ and $\hat{\mathbf{\Sigma}}_{g}$, sweep $g_{\star}$, and solve the long-only problem at each value; each point on the frontier is a different weight vector, and reading the weights along the frontier shows which assets carry it. Let's do exactly that.

> __Example__
>
> [▶ Compute minimum-variance portfolios, the efficient frontier, and the capital allocation line from data](CHEME-5660-L5b-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). Estimate the inputs for a chosen set of firms from the 2014 to 2024 data, compute the global minimum-variance portfolio in closed form and with a long-only solver, sweep the target growth rate to trace the efficient frontier, find the tangent portfolio and the capital allocation line, and compare the optimized portfolios with equal weights and an index fund on 2025 data the optimizer never saw.

___

## Adding a Risk-Free Asset
So far every asset was risky. Now add a __risk-free asset__: a horizon-matched payoff with a known growth rate $g_{f}$ (units: inverse years) and zero variance over the holding period, in practice a Treasury bill or a zero-coupon Treasury (a STRIP) held to a maturity that matches the horizon (sold earlier, it carries the interest-rate risk of L2b, so "risk-free" is a statement about the matched horizon). By definition it has zero covariance with every risky asset.

### The capital allocation line
Take any risky portfolio $p$ with expected growth rate $\mathbb{E}[g_{p}]$ and growth-rate standard deviation $\sigma_{g,p}>0$, and put a fraction $w_{f}$ of wealth in the risk-free asset and $1-w_{f}$ in $p$. Because the risk-free asset adds no variance, the resulting __complete portfolio__ $c$ has:
$$
\begin{align*}
\mathbb{E}\left[g_{c}\right] &= w_{f}\,g_{f} + (1-w_{f})\,\mathbb{E}\left[g_{p}\right] = g_{f} + (1-w_{f})\left(\mathbb{E}\left[g_{p}\right]-g_{f}\right)\\
\sigma_{g,c} &= \left|1-w_{f}\right|\,\sigma_{g,p}
\end{align*}
$$
For $w_{f}\leq1$ (a long position in the risky portfolio) we can eliminate $w_{f}$: from the second line $1-w_{f}=\sigma_{g,c}/\sigma_{g,p}$, and substituting into the first gives the __capital allocation line__ (CAL) through $p$:
$$
\boxed{
\begin{align*}
\mathbb{E}\left[g_{c}\right] &= g_{f} + \underbrace{\left(\frac{\mathbb{E}\left[g_{p}\right]-g_{f}}{\sigma_{g,p}}\right)}_{\text{Sharpe ratio}\;\text{SR}_{p}}\;\sigma_{g,c}\quad\blacksquare
\end{align*}}
$$
In the $(\sigma_{g},g)$ plane this is a straight line from the risk-free asset at $(0,g_{f})$ through $p$. Points between the two, $0<w_{f}<1$, lend part of the wealth at $g_{f}$; the point $w_{f}=0$ is $p$ itself; and points beyond $p$, $w_{f}<0$, __borrow__ at $g_{f}$ to hold more than the whole budget in $p$, which assumes we can borrow at the lending rate (an idealization; a higher borrowing rate kinks the line at $p$). The slope is the __Sharpe ratio__ of $p$, its expected excess growth rate over the risk-free rate per unit of growth-rate standard deviation.

> __Which Sharpe ratio?__ With $\sigma_{g,p}$ in the denominator, both numerator and denominator are in inverse years, so $\text{SR}_{p}$ is dimensionless, but it is the Sharpe ratio of a __one-observation-interval__ holding period, because $\sigma_{g,p}$ is the standard deviation of a one-day growth rate; for daily data its values are small. The convention in practice divides by the volatility $\sqrt{\mathbf{w}^{\top}\mathbf{C}\mathbf{w}}=\sqrt{\Delta{t}}\,\sigma_{g,p}$ instead, which gives the __annualized__ Sharpe ratio $\text{SR}_{p}/\sqrt{\Delta{t}}$, larger by $\sqrt{252}\approx15.9$. The two rank portfolios identically, so every conclusion below holds under either; the number quoted must say which. L6b onward quotes the annualized value.

### The tangent portfolio
Every risky portfolio has its own CAL, and a steeper line is better: at any standard deviation it offers more expected growth. So the risky portfolio to hold is the one whose CAL is steepest, the __tangent portfolio__ $\mathcal{T}$, which maximizes the Sharpe ratio; geometrically its CAL touches the efficient frontier at $\mathcal{T}$ and lies above the frontier everywhere else. When short positions are allowed, $\mathbf{\Sigma}_{g}$ is positive definite, and the GMV portfolio itself earns more than the risk-free rate in expectation, $\mathbb{E}[g_{\text{GMV}}]>g_{f}$ (equivalently $\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}(\bar{\mathbf{g}}-g_{f}\mathbf{1})>0$), the tangent portfolio has the closed form:
$$
\boxed{
\begin{align*}
\mathbf{w}_{\mathcal{T}} &= \frac{\mathbf{\Sigma}_{g}^{-1}\left(\bar{\mathbf{g}}-g_{f}\mathbf{1}\right)}{\mathbf{1}^{\top}\mathbf{\Sigma}_{g}^{-1}\left(\bar{\mathbf{g}}-g_{f}\mathbf{1}\right)}
\end{align*}}
$$
which is the direction $\mathbf{\Sigma}_{g}^{-1}(\bar{\mathbf{g}}-g_{f}\mathbf{1})$ scaled to sum to one (the direction comes from the same stationarity argument as the GMV derivation, applied to the excess growth rate). The sign condition matters: if the GMV portfolio earns less than $g_{f}$, the normalizing constant is negative and the same formula returns the __minimum__-Sharpe portfolio, and no fully invested tangent portfolio with a positive slope exists. Under the long-only constraint there is no closed form; the example evaluates the Sharpe ratio along the numerically computed efficient branch and takes the largest, which is accurate to the resolution of the sweep. The figure sketches the result: the risky frontier, the CAL from $g_{f}$ tangent to it at $\mathcal{T}$, and the lending and borrowing segments.

<div>
    <center>
        <img src="figs/Fig-Frontier-CAL-Schematic.png" width="640" alt="Schematic of the capital allocation line: the risky minimum-variance frontier as a curve opening to the right with the GMV portfolio at its vertex, the risk-free asset on the vertical axis, and a straight line from the risk-free asset tangent to the efficient branch at the tangent portfolio, with the segment between them labeled lending and the extension beyond labeled borrowing"/>
    </center>
</div>

> __Two-fund separation:__ Because the tangent portfolio's CAL dominates every other, all mean-variance investors facing the same estimates and the same $g_{f}$ hold the __same__ risky fund $\mathcal{T}$ and differ only in $w_{f}$: the investment decision (which risky portfolio) is separated from the financing decision (how much to lend or borrow), a result due to Tobin ([Tobin, J. (1958). Liquidity Preference as Behavior Towards Risk. The Review of Economic Studies, 25(2), 65-86](https://doi.org/10.2307/2296205)). The result is exact for the unconstrained problem with a single lending and borrowing rate, and it survives a long-only constraint on the risky weights alone: the risky fund becomes the constrained maximum-Sharpe portfolio, and investors still differ only in $w_{f}$. What breaks it is a constraint on the risk-free leg or on dollar positions: no borrowing, position limits, or a borrowing rate above the lending rate kink the attainable boundary, and then the best risky portfolio depends on the investor's position on the line, so separation holds piecewise, not globally.

___

## When Is the Tangent Portfolio the Market Portfolio?
It is tempting to call the tangent portfolio "the market". The Capital Asset Pricing Model (CAPM) does exactly that, but only under strong assumptions: all investors hold the same beliefs about $\bar{\mathbf{g}}$ and $\mathbf{\Sigma}_{g}$, optimize mean and variance over the same horizon, can lend and borrow at the same risk-free rate, and trade every asset without taxes or frictions. Then every investor holds the same tangent portfolio, and for the market to clear (every share held by someone) that common portfolio must be the __market portfolio__: all risky assets weighted by their market value. The CAL through the market portfolio is then called the __capital market line__ (CML); a CAL belongs to any risky portfolio, the CML to that one portfolio under those assumptions.

The identification is an equilibrium conclusion, not a label to attach to every maximum-Sharpe solution. An optimizer applied to a list of tickers and eleven years of estimates returns a __sample-dependent__ tangent portfolio for that universe and that model; a different universe, window, or estimator returns a different one, and none of them is the market portfolio. A broad stock index is a value-weighted portfolio of one asset class and omits many risky assets. When the example compares its tangent portfolio with an index fund, it is comparing two portfolios, not testing the CAPM.
___

## Estimation Risk Can Dominate Optimization
Both closed forms carry $\mathbf{\Sigma}_{g}^{-1}$, and the tangent portfolio also carries $\bar{\mathbf{g}}-g_{f}\mathbf{1}$. Inverting a covariance matrix amplifies its least-certain directions (small eigenvalues become large weights), and expected growth rates are the noisiest input we have: the example finds that two reasonable estimators of $\bar g_{i}$ from the same eleven years of data, the sample-mean growth rate and L4b's regression slope, differ by up to more than ten percentage points per year for the most volatile firms, and the unconstrained tangent portfolio responds to differences of that size with long and short positions larger than the whole budget. Minimum-variance weights are more stable, because they do not depend on $\bar{\mathbf{g}}$ at all, which is one reason the GMV portfolio and equal weights are common baselines.

Three practices follow. Fit the inputs only on information available before each decision, and test on data the fit never saw (the example freezes its universe and its weights at the end of 2024 and looks at 2025; one such split is a check, not a validation, and a proper backtest repeats it over many periods and includes trading costs). Compare every optimized portfolio with simple baselines such as equal weighting and the GMV portfolio. And when covariance shrinkage, weight constraints, or robust objectives are used to stabilize the solution, report them, because they change the problem being solved; the L5a advanced material on covariance estimation shows shrinkage doing exactly this to a minimum-variance portfolio out of sample.

The multiple asset GBM model of L5a supplies a separate __scenario engine__: given a fixed allocation, it simulates how wealth would evolve under the estimated joint dynamics, with prediction bands and a distribution of the scaled NPV. That is a useful check on the shape of the outcomes, but it is not independent validation when the same data estimated both the optimizer and the simulator; a realized path that lands in the tails is a warning about the estimates or the constant-parameter assumption, not proof of either.

> __Example__
>
> [▶ Simulate a portfolio with multiple asset GBM](CHEME-5660-L5b-Example-MAGBM-Portfolio-Fall-2026.ipynb). Estimate the multiple asset GBM model of L5a for a set of firms, simulate correlated 2025 price paths, compute the buy-and-hold wealth of the minimum-variance allocation on every path with prediction bands next to the realized path and an index fund, and evaluate the portfolio through the distribution of its scaled net present value.

___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L6a; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ Frontier geometry and the two-fund theorem](advanced/frontier-geometry/CHEME-5660-L5b-Advanced-FrontierGeometry-Fall-2026.ipynb). Derive the closed-form frontier hyperbola, show that every frontier portfolio is a combination of two of them, and compare the unconstrained frontier with the long-only one from the package solver.
* [▶ Estimation risk in mean-variance optimization](advanced/estimation-risk/CHEME-5660-L5b-Advanced-EstimationRisk-Fall-2026.ipynb). Resample the 2014 to 2024 growth rates to see how far the frontier, the minimum-variance weights, and the tangent weights move under sampling error, and compare their out-of-sample behavior in 2025.
___

## Summary
In this lecture, we defined a portfolio's reward and risk from its weights, mean growth-rate vector, and covariance matrix, derived the minimum-variance portfolios and the efficient frontier, and added a risk-free asset to obtain the capital allocation line, the tangent portfolio, and the conditions under which that portfolio is the market portfolio.

> __Key Takeaways:__
>
> * **Covariance determines diversification:** A portfolio's expected growth rate is the weighted average of the assets' expected growth rates, but its variance collects every pairwise covariance, so mixing assets whose growth rates are less than perfectly correlated lowers the standard deviation below the weighted average; the objective is a one-period statistic, and buy-and-hold wealth is evaluated separately.
> * **The frontier is a family of constrained minimizations:** The global minimum-variance portfolio has a closed form when short positions are allowed, the target-growth problem traces a hyperbola whose upper branch is the efficient frontier, and the long-only problem is a convex quadratic program solved numerically whose inequality target returns the minimum-variance portfolio and the efficient branch.
> * **A risk-free asset selects one risky fund:** The capital allocation line through a risky portfolio has slope equal to its Sharpe ratio, the tangent portfolio maximizes that slope so every mean-variance investor holds it and differs only in the risk-free fraction, and calling it the market portfolio requires the CAPM's common-belief, common-horizon, frictionless-trading, and market-clearing assumptions.

Next time, we replace the full covariance matrix, with its many noisy entries, by a single-index model that explains each asset's growth rate through the market's, and separates systematic from idiosyncratic risk.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___